# Working with Pandas

**Table of Content**

1. [Shortnames for imported packages](#sec1)
2. [The `DataFrame` Class](#sec2)
3. [Construct a dataframe from Python data structures](#sec3)
4. [Import external data](#sec4)
5. [Head and Tail](#sec5)
6. [The `describe` method](#sec6)
7. [Attached attributes: `dtypes`, `index`, `columns`](#sec7)
8. [Rename columns](#sec8)
9. [Sorting by one or more columns](#sec9)
10. [Indexing and Slicing](#sec10)
11. [Apply operations](#sec11)
12. [Creating new columns](#sec12)
13. [Deleting duplicates](#sec13)
14. [Filtering values](#sec14)
15. [Replacing values](#sec15)
16. [Dropping values](#sec16)
17. [A `groupby` example](#sec17)


**Credit:** Many of these examples are adapted from the extensive library of examples on data science by [Chris Albon](https://chrisalbon.com/). Visit the website for many more examples.

<a id="sec1"></a>
## 1. Shortnames for imported packages

There is a practice in Python to use shortnames for packages, instead of their full names. Also, instead of importing everything from a package, we refer to functions through the package name, to preserve the name space (i.e., avoid that functions with the same name override each other in the program's name space).

Thus, most pandas objects below are accessed as: `pd._object_name_`

It's a good idea to learn the conventional shortnames for all libraries that we'll use in the class.

In [ ]:
import pandas as pd
import numpy as np # a very useful library

<a id="sec2"></a>
## 2. The `DataFrame` Class

The most important class in `pandas` is the `DataFrame`, which represents data as a table. We will usually store in it the "tidy tables" or data that we will eventually transform into a tidy table to do the rest of the analysis.

In [ ]:
pd.DataFrame # check the nature of this object

One of the most powerful Python commands is `help`. You don't need to Google! With `help` you can get useful examples to understand a data structure, a class, a method, etc. Peruse the parameters and the examples for `DataFrame` below.

In [ ]:
help(pd.DataFrame)

**Note:** Before pushing your notebook to Github, comment out cells like the one above that print documentation through help, becuase in HTML format, they are collapsed and take up a lot of space in the noteboook.

<a id="sec3"></a>
## 3. Construct a dataframe from Python data structures

Because the dataframe is a table of rows and columns, to create one from scratch we need to supply it with two-dimensional data, for example a dictionary of lists, a list of dictionaries, a list of lists, etc.

### From a dictionary of lists

Notice here that each list associated with a key will become a column.

In [ ]:
d = {'col1': [1, 2, 3], 
     'col2': [4, 5, 6], 
     'col3': [7, 8, 9]}
d

In [ ]:
df = pd.DataFrame(data=d) # the named parameter data is optional
df

In [ ]:
# you can create the dataframe by passing the argument directly, no need for the named parameter, try it out
df = pd.DataFrame(d)

### From a list of dictionaries

Dictionaries need to have the same keys. Notice that each dictionary becomes a row in the table.

In [ ]:
d = [{'colA': 10, 'colB': 20}, 
     {'colA': 15, 'colB': 30}]
d

In [ ]:
df2 = pd.DataFrame(data=d)
df2

### From a numpy array

We will create first a 2D array (a matrix), 5x5 with random integers, using `numpy`, and then will feed that data to the DataFrame constructor.

In [ ]:
matrix = np.random.randint(low=0, high=10, 
                           size=(5, 5))
matrix                  

Because an array doesn't have keys that can be used as column names (see the first two examples), we need to supply them through the named parameter `columns`, see below:

In [ ]:
df3 = pd.DataFrame(matrix, columns=['a', 'b', 'c', 'd', 'e'])
df3

### From a list of lists (with indices and columns as parameters)

If we don't like it that the rows have indices like 0, 1, 2, etc., we can provide our own indices. Below, we first create a list of indices and columns, and then use them as values for the parameters `index` and `columns` in the `DataFrame` constructor call.

In [ ]:
import string
indices = list(string.ascii_uppercase) # all the uppercase letters
columns = list(string.ascii_lowercase) # all the lowercase letters

print(indices)
print(columns)

In [ ]:
df4 = pd.DataFrame(np.random.randint(low=0, high=100, size=(10, 10)),
                               index=indices[:10], 
                               columns=columns[:10])
df4

As you can see, the `DataFrame` constructor is very flexible. The first parameter refers to the data that we want to store in it, but then we can provide values for the other parameters, referring them through names.

<a id="sec4"></a>
## 4. Import external data

Very frequently, we will be importing data into `pandas` from external files.

In the row below, put the cursor at the end and press the key _tab_ to see the many `read` methods that allow you to import various data formats:

In [ ]:
pd.read_

**Read from a CSV file**

The most common file from which data is imported is the CSV. For the example below, we will use a CSV file with responses from a survey of CS 234 students in 2019.

In [ ]:
dfStudents = pd.read_csv('survey-responses.csv')
dfStudents

<a id="sec5"></a>
## 5. Head and Tail

You can use these two commands (`head`, `tail`) to print only a portion of the dataframe, this is useful when the dataframe has too many rows and you only want to look at a few.

In [ ]:
dfStudents.head()

By default, the first (or last) five rows will be printed. But, we can control the number of printed rows:

In [ ]:
dfStudents.tail(3)

<a id="sec6"></a>
## 6. Attached attributes

The attributes `dtypes`, `index` and `columns` can be accessed individually to learn more about the dataframe:

In [ ]:
dfStudents.dtypes # describes the datatype for each column
# this is helpful to know what operations you can apply on each column

In [ ]:
dfStudents.index # often, this shows the automatic indices assigned to the dataframe

In [ ]:
df4.index # this is for the dataframe we created with our own indices

In [ ]:
dfStudents.columns

<a id="sec7"></a>
## 7. The `describe` method

This method provides descriptive statistics for every quantitative variable of the dataframe. It's the best way to explore the dataset in one step. Generating descriptive statistics is one of the first steps in data exploration.

In [ ]:
dfStudents.describe()

When applied on the dataframe, the method calculates the descriptive statistics for every variable that is quantitative (in the case above, 4 variables).

One can use the method for qualitative variables as well, but the returned descriptive statistics are different. See an example below:

In [ ]:
dfStudents['Your favorite drink:'].describe()

<a id="sec8"></a>
## 8. Rename columns in a dataframe

The column names of a survey are long strings, and hard to distinguish. It is better to have some short names, which then are easy to use when we want to refer to columns in our code. To change the names, we can create a dict of pairs mapping old names to new ones and then using the `rename` method by passing the dict as a parameter.

In [ ]:
names = {'Your class year:': 'classyear', 
         'Your favorite number:': 'fav_num',
         'Your height in centimeters:': 'height', 
         'Your weight in kilogram:': 'weight',
         'Your favorite drink:': 'fav_drink', 
         'What is your favorite animal?': 'fav_animal',
         'What is your favorite movie?': 'fav_movie', 
         'Who is your favorite woman author?': 'fav_author'}

dfStudents.rename(columns=names)

We can see that the column names have changed, but is that change permanent? We can print the dataframe again:

In [ ]:
dfStudents.head()

**What is going on, why do we still have the old names?**

When we invoke a method on a dataframe object, a new object is created and the old one is preserved unchanged. We have two options at this point:

- use a new variable to store the result of a method
- change the original object

Most dataframe objects have a boolean parameter `inplace`, which controls whether the manipulation should happen to the current object or not. We can verify that with the help command:

In [ ]:
help(dfStudents.rename)

Now we can try it out, we will use `inplace` instead of creating a new dataframe object. Calling `head` let's us verify that the change happened.

In [ ]:
dfStudents.rename(columns=names, inplace=True)
dfStudents.head()

<a id="sec9"></a>
## 9. Sorting by one or more columns

Let's look at the various columns and pick one to sort by:

In [ ]:
dfStudents.head()

Let's sort by **height**, by simply providing its name as a parameter to the `sort_values` method. Also, notice that we can chain multiple method calls, as shown below:

In [ ]:
dfStudents.sort_values("height").head()

Does `sort_values` change the current dataframe? We should check:

In [ ]:
dfStudents.head()

It did not. Thus, yet another example that methods create a new dataframe object, and you have to make a choice of whether to store the result into a new variable, or use `inplace=True` to change the current object. 

Always use `help` to check what parameters a method provides.

In [ ]:
help(dfStudents.sort_values)

Now that we have seen what is possible, we try out some of the parameters ourselves, for example, ascending, and multiple columns:

In [ ]:
dfStudents.sort_values(['fav_num', 'height'], ascending=False)

**Sorting by rows**

We most commonly will sort by columns, but in pandas it's possible to sort by a row too, this will change the order of the columns as a result. Let's try it out with df4, in which all values are numerical:

In [ ]:
df4

In [ ]:
df4.sort_values('A', axis=1) # we need to set axis=1, in order to sort by row

Notice that the values in row A are sorted from the smallest to the largest, and the column names are in different oroder.

<a id="sec10"></a>
## 10. Indexing and Slicing

One of the most common operations that we might want to perform is slicing, which allows us to select only some desired values such as:
- a single column
- several columns
- a single row
- several rows
- a subtable (a few rows and a few columns)
- a single value

Let's refer to a simpler dataframe to work with, for example, `df4`:

In [ ]:
df4

### Two ways to get a single column

- use column name as attribute name: df.colname
- use column name in brackets: df['colname']

In [ ]:
df4.a

In [ ]:
df4['b']

### Several columns

Use a list of names to get multiple columns. We'll end up with double brackets:
- outer brackets are the subscription operator
- inner brackets are for the list of column names

In [ ]:
df4[['a', 'c', 'd']]

The order of columns can be different from the original, so we can create a different dataframe, if we want:

In [ ]:
df5 = df4[['g', 'e', 'a']]
df5.head()

However, the order of the indices (rows) remains the same:

In [ ]:
df5.index

### Accessing single rows

A common mistake is to think that just like we accessed single columns by using the column name, we can do the same by using the index label, or its index. However, such efforts will cause a KeyError:

In [ ]:
df5['A']

In [ ]:
df5[2]

In the following, we show various ways to overcome this error using either the slice operator (:), or the two dedicated methods `loc` and `iloc`.

### Select many entire rows via indices or labels

We can use the slice operator to get several rows:

In [ ]:
df5[1:4]

But, labels can also be used, together with the slicing operator, as in:

In [ ]:
df5[:'A']

You can get all rows up (and including) a given label:

In [ ]:
df5[:'C']

Or provide a range of rows:

In [ ]:
df5['D':'G']

Notice that with labels, the slicing operator is inclusive. This is not the case for indices:

In [ ]:
df5[2:4] # shows only rows 2 and 3, not, 2, 3, 4.

### Operators `loc` and `iloc` to access groups of rows & columns or single cells

Both these operators have similar functionality, but `loc` uses labels for indexing, while `iloc` uses integers. As for the other operations, I recommend that you read the documentation via `help` for more clarification and examples.

#### Using `loc`

In [ ]:
# try this, if you are curious about loc
help(df5.loc)

In [ ]:
# one row
df5.loc['A']

If the output of an operation looks unfamiliar, use `type` to check what kind of value it is, in this case, instead of a dataframe, we received an object of type `Series`:

In [ ]:
type(df5.loc['A'])

We can use `loc` to access multiple rows:

In [ ]:
# multiple rows
df5.loc[['A','C']] # list of index labels

In [ ]:
# multiple rows and one column
df5.loc['A': 'C', 'e']

In [ ]:
# one row two columns
df5.loc['A', ['e','a']]

In [ ]:
# multiple rows, multiple columns
df5.loc['A':'C', ['e','a']]

#### Use `iloc`

In [ ]:
help(df5.iloc)

In [ ]:
# access one row
df5.iloc[0]

In [ ]:
# access many rows
df5.iloc[2:5]

In [ ]:
# access one column
df5.iloc[:,2]

In [ ]:
# access many columns
df5.iloc[:,1:3]

In [ ]:
# access few rows and few colums
df5.iloc[1:3,1:3]

### How to access a single value in the dataframe

Let's see again the whole dataframe:

In [ ]:
df5

We can use labels (names for rows and columns) together to locate a single cell in the dataframe:

In [ ]:
df5.loc['A', 'a']  # the cell at the intersection of 'A' and 'a'

Or we can use indices for both rows and columns:

In [ ]:
df5.iloc[0, 2] # item in 0th row, 2nd column

<a id="sec11"></a>
## 11. Apply operations

In pandas, we often need to apply an operation to all members of a column or the whole dataframe. The methods `apply` and `applymap` will be helpful in that regard.

Let's create a dataframe from a dictionary first, to use in the examples:

In [ ]:
data = {'name': ['Jason', 'Molly', 'Tina', 'Jake', 'Amy'], 
        'year': [2012, 2012, 2013, 2014, 2014], 
        'reports': [4, 24, 31, 2, 3],
        'coverage': [25, 94, 57, 62, 70]}

df = pd.DataFrame(data, index = ['Cochice', 'Pima', 'Santa Cruz', 'Maricopa', 'Yuma'])
df

### Capitalize all names in column `name`

Initially, we create a lambda function and give it a name:

In [ ]:
capitalizer = lambda x: x.upper()

Let's test that it works:

In [ ]:
capitalizer("wellesley")

We now use the method `apply` on the column `name`, by passing the new function as an argument:

In [ ]:
df['name'].apply(capitalizer)

However, such operation hasn't changed the dataframe:

In [ ]:
df

To do that, we can assign the new column to the old one:

In [ ]:
df['name'] = df['name'].apply(capitalizer)
df

**YOUR TURN:** Increase the `year` values by five using the approach shown above.

In [ ]:
# write your solution here


### Change all elements
Writing more sophisticated functions allows to apply an operation to almost all values in the dataframe. The function below multiplies by 100 every numerical value, but doesn't touch the string elements.

This function is an element-wise function, intended to be used on all elements of a dataframe, instead of an entire column, or entire row.

In [ ]:
# create a function called times100
def times100(x):
    # that, if x is a string,
    if type(x) is str:
        # just returns it untouched
        return x
    # but, if not, return it multiplied by 100
    elif x:
        return 100 * x
    # and leave everything else
    else:
        return

In [ ]:
df.applymap(times100)

<a id="sec12"></a>
## 12. Creating new columns

It's possible to add new columns to a dataframe at any time. Here are two ways to do that:

In [ ]:
# 1. use the subscription operator []

df = pd.DataFrame()
df['name'] = ['ben', 'lyn', 'ada']
df

In [ ]:
# 2. use the method assign

df = df.assign(lastname=['wood','turbak','lerner'])
df

<a id="sec13"></a>
## 13. Deleting duplicates

Sometimes there are identical rows, and sometimes there are rows that are partially identical. The example below shows how to remove them.

In [ ]:
raw_data = {'first_name': ['Jason', 'Jason', 'Jason','Tina', 'Jake', 'Amy'], 
        'last_name': ['Miller', 'Miller', 'Miller','Ali', 'Milner', 'Cooze'], 
        'age': [42, 42, 1111111, 36, 24, 73], 
        'preTestScore': [4, 4, 4, 31, 2, 3],
        'postTestScore': [25, 25, 25, 57, 62, 70]}
df = pd.DataFrame(raw_data, columns = ['first_name', 'last_name', 'age', 'preTestScore', 'postTestScore'])
df

Notice that we have two identical rows, 0 and 1, and one that is partially identical to them (row 2).

First, one can check for duplicate rows:

In [ ]:
df.duplicated()

The result shows that row 1 is a duplicate. It's easy to drop identical duplicates:

In [ ]:
df.drop_duplicates()

Notice that row 1 disappeared. Meeanwhile, the same method, if used with named arguments, can remove partial duplicates:

In [ ]:
df.drop_duplicates(['first_name'], keep='last')

That is, we removed rows with the same first_name, keeping in the table the last element (by default, this method keeps the first occurrence of a duplicate).

<a id="sec14"></a>
## 14. Filtering values

One way to "filter" data is to select only a few columns or a few rows, as we saw when we learned to slice a dataframe. However, often we want to filter based on the values of variables, in that case, we are going to create boolean expressions that will keep some values and ignore some others, as the examples in this section show.

In [ ]:
data = {'name': ['Jason', 'Molly', 'Tina', 'Jake', 'Amy'], 
        'year': [2012, 2012, 2013, 2014, 2014], 
        'reports': [4, 24, 31, 2, 3],
        'coverage': [25, 94, 57, 62, 70]}
df = pd.DataFrame(data, index = ['Cochice', 'Pima', 'Santa Cruz', 'Maricopa', 'Yuma'])
df

**Find rows where `coverage` is greater than 50**

In [ ]:
df[df['coverage'] > 50] # notice the boolean expression within the brackets

**A complex filtering expression**

In [ ]:
df[(df['coverage']  > 50) & (df['reports'] < 4)] # two combined boolean expressions

**Note:** The operation above used the operator `&` to combine two boolean expressions, that is different from Python's expressions that use the operator `&&`.

<a id="sec15"></a>
## 15. Replacing values

Sometimes, we want to replace many values at once. That is easy to do with the `replace` method.

In [ ]:
raw_data = {'first_name': ['Jason', 'Molly', 'Tina', 'Jake', 'Amy'], 
        'last_name': ['Miller', 'Jacobson', 'Ali', 'Milner', 'Cooze'], 
        'age': [42, 52, 36, 24, 73], 
        'preTestScore': [-999, -999, -999, 2, 1],
        'postTestScore': [2, 2, -999, 2, -999]}
df = pd.DataFrame(raw_data)
                  
df

In [ ]:
df.replace(-999, np.nan) # np.nan means "Not a Number", a common way of denoting missing values in a dataset

As always though, this method returns a new dataframe. If you want to save the changes, you can either:
- store the result into a new variable
- use the `inplace` parameter to change the exisiting datafram

<a id="sec16"></a>
## 16. Dropping values

We might want to drop entire columns or rows, we can do that with the method `drop`. Study its documentation to learn about its different parameters.

In [ ]:
data = {'name': ['Jason', 'Molly', 'Tina', 'Jake', 'Amy'], 
        'year': [2012, 2012, 2013, 2014, 2014], 
        'reports': [4, 24, 31, 2, 3]}
df = pd.DataFrame(data, index = ['Cochice', 'Pima', 'Santa Cruz', 'Maricopa', 'Yuma'])
df

**Drop rows (observations)**

In [ ]:
df.drop(['Cochice', 'Pima'])

**Drop columns(variables)**

In [ ]:
df.drop('reports', axis=1) # axis=1 denotes that we are referring to a column, not a row

**Note:** Both examples above didn't change the original frame. To do that, use `inplace=True`.

<a id="sec17"></a>
## 17. A simple `groupby` example

This is just a simple taste of this powerful method. We will keep talking about it in future assignments.

In [ ]:
# Example dataframe
raw_data = {'regiment': ['Nighthawks', 'Nighthawks', 'Nighthawks', 'Nighthawks', 'Dragoons', 'Dragoons', 'Dragoons', 'Dragoons', 'Scouts', 'Scouts', 'Scouts', 'Scouts'], 
        'company': ['1st', '1st', '2nd', '2nd', '1st', '1st', '2nd', '2nd','1st', '1st', '2nd', '2nd'], 
        'name': ['Miller', 'Jacobson', 'Ali', 'Milner', 'Cooze', 'Jacon', 'Ryaner', 'Sone', 'Sloan', 'Piger', 'Riani', 'Ali'], 
        'preTestScore': [4, 24, 31, 2, 3, 4, 24, 31, 2, 3, 2, 3],
        'postTestScore': [25, 94, 57, 62, 70, 25, 94, 57, 62, 70, 62, 70]}
df = pd.DataFrame(raw_data, columns = ['regiment', 'company', 'name', 'preTestScore', 'postTestScore'])
df

In [ ]:
# Create a grouping object. In other words, create an object that
# represents that particular grouping. In this case we group
# pre-test scores by the regiment.
regiment_preScore = df['preTestScore'].groupby(df['regiment'])
regiment_preScore

In [ ]:
# Display the mean value of the each regiment's pre-test score
regiment_preScore.mean()

In [ ]:
# Display the max value of the each regiment's pre-test score
regiment_preScore.max()

In [ ]:
# Display the total count of the each regiment
regiment_preScore.count()

The idea is that once we have grouped the data by a certain variable, we can apply many variuos statistical operations to the groups.

For a very detailed and long explanation of `groupBy`, you can read [this page](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html).